In [4]:
"""
DATABASE LAYER — SQLite-backed application and audit storage.
"""
import sqlite3, json, os
from datetime import datetime
from pathlib import Path
from typing import Optional, List


BASE_DIR = Path.cwd()

DB_PATH = BASE_DIR / "data" / "lending_platform.db"


def get_connection() -> sqlite3.Connection:
    Path(os.path.dirname(DB_PATH)).mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_connection()
    c = conn.cursor()

    c.execute("""
    CREATE TABLE IF NOT EXISTS applications (
        id            INTEGER PRIMARY KEY AUTOINCREMENT,
        application_id TEXT UNIQUE NOT NULL,
        timestamp      TEXT,
        -- Borrower info
        monthly_income REAL,
        employment_type TEXT,
        credit_score    INTEGER,
        age             INTEGER,
        city            TEXT,
        gender          TEXT,
        -- Loan request
        loan_amount     REAL,
        loan_tenure_months INTEGER,
        loan_purpose    TEXT,
        -- Credit signals
        existing_loans  INTEGER,
        missed_payment_ratio REAL,
        debt_to_income_ratio REAL,
        worst_delinquency_stage INTEGER,
        -- Decision outputs
        pd_score        REAL,
        risk_grade      TEXT,
        decision        TEXT,
        confidence      REAL,
        interest_rate   REAL,
        approved_amount REAL,
        emi_amount      REAL,
        expected_loss   REAL,
        fraud_risk_score REAL,
        fraud_severity  TEXT,
        health_score    REAL,
        health_ladder   TEXT,
        -- Full JSON blobs
        decision_json   TEXT,
        status          TEXT DEFAULT 'Submitted',
        -- Underwriter actions
        underwriter_notes TEXT,
        override_decision TEXT,
        override_reason   TEXT,
        reviewed_at       TEXT,
        reviewed_by       TEXT
    )
    """)

    c.execute("""
    CREATE TABLE IF NOT EXISTS audit_log (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp   TEXT,
        event_type  TEXT,
        application_id TEXT,
        user_role   TEXT,
        action      TEXT,
        details     TEXT
    )
    """)

    c.execute("""
    CREATE TABLE IF NOT EXISTS portfolio_snapshots (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        snapshot_date TEXT,
        total_loans INTEGER,
        total_aum   REAL,
        avg_pd      REAL,
        avg_raroc   REAL,
        ecl_total   REAL,
        approved_count INTEGER,
        declined_count INTEGER,
        fraud_flagged  INTEGER,
        snapshot_json  TEXT
    )
    """)

    conn.commit()
    conn.close()


def save_application(decision_obj, app_input) -> int:
    conn = get_connection()
    c    = conn.cursor()

    # Serialise the full decision object
    decision_dict = {
        k: v for k, v in decision_obj.__dict__.items()
    }
    decision_dict["top_risk_factors"] = decision_obj.top_risk_factors
    decision_dict["protective_factors"]= decision_obj.protective_factors

    try:
        c.execute("""
        INSERT OR REPLACE INTO applications
        (application_id, timestamp,
         monthly_income, employment_type, credit_score, age, city, gender,
         loan_amount, loan_tenure_months, loan_purpose,
         existing_loans, missed_payment_ratio, debt_to_income_ratio,
         worst_delinquency_stage,
         pd_score, risk_grade, decision, confidence,
         interest_rate, approved_amount, emi_amount, expected_loss,
         fraud_risk_score, fraud_severity, health_score, health_ladder,
         decision_json, status)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            decision_obj.application_id,
            decision_obj.timestamp,
            app_input.monthly_income, app_input.employment_type,
            app_input.credit_score, app_input.age,
            app_input.city, app_input.gender,
            app_input.loan_amount, app_input.loan_tenure_months,
            app_input.loan_purpose,
            app_input.existing_loans, app_input.missed_payment_ratio,
            app_input.debt_to_income_ratio, app_input.worst_delinquency_stage,
            decision_obj.pd_score, decision_obj.risk_grade,
            decision_obj.decision, decision_obj.confidence,
            decision_obj.interest_rate, decision_obj.approved_amount,
            decision_obj.emi_amount, decision_obj.expected_loss,
            decision_obj.fraud_risk_score, decision_obj.fraud_severity,
            decision_obj.health_score, decision_obj.health_ladder,
            json.dumps(decision_dict),
            "Submitted",
        ))
        row_id = c.lastrowid
        conn.commit()
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()
    return row_id


def get_application(application_id: str) -> Optional[dict]:
    conn = get_connection()
    c    = conn.cursor()
    c.execute("SELECT * FROM applications WHERE application_id = ?", (application_id,))
    row  = c.fetchone()
    conn.close()
    return dict(row) if row else None


def get_all_applications(limit: int = 500) -> List[dict]:
    conn = get_connection()
    c    = conn.cursor()
    c.execute(
        "SELECT * FROM applications ORDER BY id DESC LIMIT ?", (limit,)
    )
    rows = [dict(r) for r in c.fetchall()]
    conn.close()
    return rows


def update_application_status(application_id: str, status: str,
                               notes: str = "", override: str = "",
                               reason: str = "", reviewer: str = ""):
    conn = get_connection()
    c    = conn.cursor()
    c.execute("""
    UPDATE applications
    SET status = ?,
        underwriter_notes = ?,
        override_decision = ?,
        override_reason   = ?,
        reviewed_at       = ?,
        reviewed_by       = ?
    WHERE application_id = ?
    """, (status, notes, override, reason, datetime.now().isoformat(), reviewer, application_id))
    conn.commit()
    conn.close()


def log_audit(event_type: str, application_id: str, role: str,
              action: str, details: str = ""):
    conn = get_connection()
    c    = conn.cursor()
    c.execute("""
    INSERT INTO audit_log (timestamp, event_type, application_id, user_role, action, details)
    VALUES (?,?,?,?,?,?)
    """, (datetime.now().isoformat(), event_type, application_id, role, action, details))
    conn.commit()
    conn.close()


def get_portfolio_stats() -> dict:
    conn = get_connection()
    c    = conn.cursor()
    c.execute("""
    SELECT
        COUNT(*) as total,
        SUM(loan_amount) as total_aum,
        AVG(pd_score) as avg_pd,
        SUM(expected_loss) as total_ecl,
        SUM(CASE WHEN decision='APPROVE' THEN 1 ELSE 0 END) as approved,
        SUM(CASE WHEN decision='DECLINE' THEN 1 ELSE 0 END) as declined,
        SUM(CASE WHEN decision='MANUAL_REVIEW' THEN 1 ELSE 0 END) as manual,
        SUM(CASE WHEN fraud_severity IN ('Red','Orange') THEN 1 ELSE 0 END) as fraud_flagged,
        AVG(interest_rate) as avg_rate,
        AVG(health_score) as avg_health
    FROM applications
    """)
    row = c.fetchone()
    conn.close()
    return dict(row) if row else {}


# Initialise on import
init_db()